# Alpha Research Pipeline

7-step template: Signal -> Rank -> Position -> Shift -> Returns -> Combine -> Evaluate

Uses `data_loader.py` and `alpha_utils.py` — no logic lives in this notebook.
This notebook is just: load data -> define signal -> call `evaluate_alpha()` -> plot.

In [1]:
import matplotlib.pyplot as plt
from data_loader import load_data, get_close_prices, get_volume, get_open_prices
from alpha_utils import evaluate_alpha
from alpha_utils import ts_rank, ts_min, ts_max
from alpha_utils import rolling_corr
import numpy as np 
import pandas as pd 
import warnings
from scipy.stats import ConstantInputWarning
warnings.filterwarnings('ignore', category=ConstantInputWarning)

data = load_data()
close = get_close_prices(data)
open_px = get_open_prices(data)
volume = get_volume(data)
n_stocks = close.shape[1]
print(close.shape)

Loading cached data from data/sp500_5y.parquet
(1255, 503)


In [2]:
print(data.columns.get_level_values(1).unique())

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close'], dtype='object', name='Price')


## Alpha 1 — 21-day momentum

In [3]:
signal_alpha1 = close.pct_change(21, fill_method=None)
results_1 = evaluate_alpha(signal_alpha1, close, n_stocks)

print("Sharpe:", results_1["sharpe"])
print("IC mean:", results_1["ic_mean"])
print("ICIR:", results_1["icir"])

Sharpe: -0.10054801406387627
IC mean: -0.006177787063267017
ICIR: -0.030536156319343456


## Alpha 2 — Volume-weighted price trend

In [4]:
price_trend = close.pct_change(21, fill_method=None)
vol_weight = volume.rolling(21).mean()
signal_alpha2 = price_trend * vol_weight

results_2 = evaluate_alpha(signal_alpha2, close, n_stocks)

print("Sharpe:", results_2["sharpe"])
print("IC mean:", results_2["ic_mean"])
print("ICIR:", results_2["icir"])

Sharpe: 0.1775904676405182
IC mean: -0.00538666960354889
ICIR: -0.029074522629863705


## Aplha 3 - Mean Reversion

In [5]:
open_prices = data.xs('Open', axis=1, level='Price')

In [6]:
rank_open = open_prices.rank(axis=1)
rank_volume = volume.rank(axis=1)
roll_corr = rank_open.rolling(10).corr(rank_volume)
signal_alpha3 = -1 * (roll_corr)

In [7]:
results_3 = evaluate_alpha(signal_alpha3, close, n_stocks)

print('Sharpe:', results_3['sharpe'])
print('IC mean', results_3['ic_mean'])
print('ICIR', results_3['icir']) 

Sharpe: 0.22412629089611272
IC mean 0.002694015931150158
ICIR 0.04199867160359809


## Alpha 4 - Mean Reversion (Price Trend Relationship)

In [8]:
low_prices = data.xs('Low', axis=1, level='Price')

In [9]:
rank_low = low_prices.rank(axis=1)
ts_rank_low = ts_rank(rank_low, 9)
signal_alpha_4 = -1 * ts_rank_low

In [10]:
results_4 = evaluate_alpha(signal_alpha_4, close, n_stocks)
print('Sharpe:', results_4['sharpe'])
print('IC mean', results_4['ic_mean'])
print('ICIR', results_4['icir'])

Sharpe: 0.5890015063792607
IC mean 0.010448494449174827
ICIR 0.06237574512406685


## Alpha 6 - Mean Reversion (Price Volume Relationship) 

In [11]:
volume = data.xs("Volume", axis=1, level='Price')

In [12]:
signal_alpha5 = -1 * rolling_corr(open_prices, volume, 10)

In [13]:
results_5 = evaluate_alpha(signal_alpha5, close, n_stocks)

print('Sharpe:', results_5['sharpe'])
print('IC mean', results_5['ic_mean'])
print('ICIR', results_5['icir'])

Sharpe: 0.029584935399746083
IC mean 0.000843148185622812
ICIR 0.007474413117606575


## Alpha 7 Volume Conditioned Momentum

In [14]:
adv20 = volume.rolling(20).mean()
delta_close7 = close.diff(7)
abs_delta = delta_close7.abs()

ts_rank60 = ts_rank(abs_delta, 60)
sign_delta = np.sign(delta_close7)

condition = adv20 < volume 

In [15]:
signal_alpha6 = np.where(condition, -1 * ts_rank60 * sign_delta, -1)
signal_alpha6 = pd.DataFrame(signal_alpha6, index=close.index, columns=close.columns)

In [16]:
results_6 = evaluate_alpha(signal_alpha6, close, n_stocks)

print('Sharpe:', results_6['sharpe'])
print('IC mean', results_6['ic_mean'])
print('ICIR', results_6['icir'])

Sharpe: 0.18794037417073506
IC mean 0.007925097247686955
ICIR 0.06300665048549996


## Alpha 8

In [17]:
returns = close.pct_change()

/var/folders/76/6k6n689j2g5gzbx7kpxdvjl00000gn/T/ipykernel_4413/3478380900.py:1: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = close.pct_change()


In [18]:
sum_open5 = open_px.rolling(5).sum()
sum_returns5 = returns.rolling(5).sum()

In [19]:
product = sum_open5 * sum_returns5
delta_10 = product - product.shift(10) 

In [20]:
signal_alpha7 = -1 * delta_10.rank(axis=1)

In [21]:
results_7 = evaluate_alpha(signal_alpha7, close, n_stocks)
print('Sharpe:', results_7['sharpe'])
print('IC mean', results_7['ic_mean'])
print('ICIR', results_7['icir'])

Sharpe: 0.6695184744068113
IC mean 0.008550690291864586
ICIR 0.05332444049079418


## Alpha 12

In [22]:
close_prices = get_close_prices(data)

In [23]:
delta1 = -1 * close_prices.diff(1)
delta_volume = volume.diff(1)
sign_delta = np.sign(delta_volume)

In [24]:
signal_alpha8 = sign_delta * delta1

In [25]:
results_8 = evaluate_alpha(signal_alpha8, close, n_stocks)
print('Sharpe:', results_8['sharpe'])
print('IC mean', results_8['ic_mean'])
print('ICIR', results_8['icir']) 

Sharpe: -0.3254798044366002
IC mean 0.0036506151025380634
ICIR 0.0410136775789619


## Alpha 9 - Regime Switch Alpha 

In [26]:
delta_close1 = close_prices.diff(1)
ts_min5 = ts_min(delta_close1, 5)
ts_max5 = ts_max(delta_close1, 5)

condition_up = ts_min5 > 0
condition_down = ts_max5 < 0 

In [27]:
inner = np.where(condition_down, delta_close1, -1 * delta_close1)
signal_alpha9 = np.where(condition_up, inner, -1 * delta_close1)
signal_alpha9 = pd.DataFrame(signal_alpha9, index=delta_close1.index, columns=delta_close1.columns)

In [28]:
results_9 = evaluate_alpha(signal_alpha9, close, n_stocks)
print('Sharpe:', results_9['sharpe'])
print('IC mean', results_9['ic_mean'])
print('ICIR', results_9['icir']) 

Sharpe: 0.537918997221931
IC mean 0.012874253239078093
ICIR 0.07718530244326223


## Alpha 15 Cross - Sectional Rank

In [29]:
high_prices = data.xs('High', axis=1, level='Price')

In [30]:
rank_high = high_prices.rank(axis=1, method='first')
rank_volume = volume.rank(axis=1, method='first')

corr = rank_high.rolling(3).corr(rank_volume)

rank_corr = corr.rank(axis=1, method='first')

In [31]:
signal_alpha15 = -1 * rank_corr.rolling(3).sum()

In [32]:
results_15 = evaluate_alpha(signal_alpha15, close, n_stocks)

print('Sharpe:', results_15['sharpe'])
print('IC mean', results_15['ic_mean'])
print('ICIR', results_15['icir']) 

Sharpe: -0.1151374123971021
IC mean 0.0012060250820195753
ICIR 0.018609168733224248
